# EDA: StatCan LFS table 14100287

Goal: understand the raw data well enough to define the Bronze load and the Silver model.

Source: `data/raw/14100287-eng.zip`, downloaded from StatCan.

In [1]:
import zipfile

import pandas as pd

pd.set_option("display.max_columns", 30)
pd.set_option("display.width", 200)

ZIP_PATH = "../data/raw/14100287-eng.zip"
DATA_FILE = "14100287.csv"
META_FILE = "14100287_MetaData.csv"

## 1. ZIP contents

The ZIP has two files, so `pd.read_csv` can't read it directly. We open it with `zipfile` and pass the file handle to pandas.

In [2]:
with zipfile.ZipFile(ZIP_PATH) as z:
    for info in z.infolist():
        print(f"{info.filename}: {info.file_size / 1e6:.1f} MB")

14100287.csv: 1180.0 MB
14100287_MetaData.csv: 0.0 MB


## 2. Quick look at the raw rows

Reading only a few rows first, everything as text, to see the data exactly as published.

In [3]:
with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(DATA_FILE) as f:
        sample = pd.read_csv(f, nrows=5, dtype=str)

sample.T

,0,1,2,3,4
REF_DATE,1976-01,1976-01,1976-01,1976-01,1976-01
GEO,Canada,Canada,Canada,Canada,Canada
DGUID,2021A000011124,2021A000011124,2021A000011124,2021A000011124,2021A000011124
Labour force characteristics,Population,Population,Population,Population,Population
Gender,Total - Gender,Total - Gender,Total - Gender,Total - Gender,Total - Gender
Age group,15 years and over,15 years and over,15 to 64 years,15 to 64 years,15 to 24 years
Statistics,Estimate,Estimate,Estimate,Estimate,Estimate
Data type,Seasonally adjusted,Unadjusted,Seasonally adjusted,Unadjusted,Seasonally adjusted
UOM,Persons in thousands,Persons in thousands,Persons in thousands,Persons in thousands,Persons in thousands
UOM_ID,428,428,428,428,428


## 3. Full load

The CSV is ~1.2 GB. Loaded as plain strings it takes ~6 GB of RAM. Text columns repeat the same few values millions of times, so `category` dtype cuts memory a lot.

In [4]:
dtypes = {col: "category" for col in sample.columns}
dtypes["VALUE"] = "float64"
dtypes["REF_DATE"] = "str"

with zipfile.ZipFile(ZIP_PATH) as z:
    with z.open(DATA_FILE) as f:
        df = pd.read_csv(f, dtype=dtypes, low_memory=False)

print(f"rows: {len(df):,}  columns: {df.shape[1]}")
print(f"memory: {df.memory_usage(deep=True).sum() / 1e9:.2f} GB")

rows: 5,466,528  columns: 19
memory: 0.46 GB


In [5]:
pd.DataFrame({
    "nunique": df.nunique(),
    "nulls": df.isna().sum(),
})

,nunique,nulls
REF_DATE,608,0
GEO,11,0
DGUID,11,0
Labour force characteristics,9,0
Gender,3,0
Age group,9,0
Statistics,4,0
Data type,3,0
UOM,2,0
UOM_ID,2,0


## 4. Dimensions

In [6]:
dimensions = [
    "GEO",
    "Labour force characteristics",
    "Gender",
    "Age group",
    "Statistics",
    "Data type",
]

for col in dimensions:
    print(df[col].value_counts(), "\n")

GEO
Canada                       632928
Alberta                      483360
British Columbia             483360
Manitoba                     483360
New Brunswick                483360
Newfoundland and Labrador    483360
Nova Scotia                  483360
Ontario                      483360
Prince Edward Island         483360
Quebec                       483360
Saskatchewan                 483360
Name: count, dtype: int64 

Labour force characteristics
Employment              672448
Employment rate         665760
Labour force            665760
Participation rate      665760
Unemployment            665760
Unemployment rate       665760
Full-time employment    588544
Part-time employment    588544
Population              288192
Name: count, dtype: int64 

Gender
Total - Gender    1862304
Men+              1802112
Women+            1802112
Name: count, dtype: int64 

Age group
15 years and over    849376
15 to 24 years       842688
15 to 64 years       842688
25 to 54 years       842688
2

## 5. Time coverage

In [7]:
print("first month:", df["REF_DATE"].min())
print("last month: ", df["REF_DATE"].max())
print("months:     ", df["REF_DATE"].nunique())

rows_per_month = df.groupby("REF_DATE", observed=True).size()
rows_per_month.describe()

first month: 1976-01


last month:  2026-08
months:      608


count     608.0
mean     8991.0
std         0.0
min      8991.0
25%      8991.0
50%      8991.0
75%      8991.0
max      8991.0
dtype: float64

Every month has exactly the same number of rows (8,991), one per series. All series run the full period from 1976-01 to the latest month.

## 6. Grain: what identifies a row?

In [8]:
print("VECTOR + REF_DATE duplicates:", df.duplicated(["VECTOR", "REF_DATE"]).sum())
print("dimensions + REF_DATE duplicates:", df.duplicated(dimensions + ["REF_DATE"]).sum())
print("distinct VECTOR:", df["VECTOR"].nunique())
print("distinct COORDINATE:", df["COORDINATE"].nunique())

VECTOR + REF_DATE duplicates: 0


dimensions + REF_DATE duplicates: 0
distinct VECTOR: 8991
distinct COORDINATE: 8991


`VECTOR` is StatCan's ID for one time series (one combination of the 6 dimensions). `VECTOR + REF_DATE` is unique, so that is the natural key. `COORDINATE` is a 1:1 alternative ID for the same series.

## 7. Units and scale

In [9]:
pd.crosstab(df["UOM"], df["SCALAR_FACTOR"])

SCALAR_FACTOR,thousands,units
UOM,,
Percent,0,1997280
Persons in thousands,3469248,0


In [10]:
pd.crosstab(df["Labour force characteristics"], df["UOM"])

UOM,Percent,Persons in thousands
Labour force characteristics,,
Employment,0,672448
Employment rate,665760,0
Full-time employment,0,588544
Labour force,0,665760
Part-time employment,0,588544
Participation rate,665760,0
Population,0,288192
Unemployment,0,665760
Unemployment rate,665760,0


Important: **only the person counts are in thousands**. The rates (unemployment, participation, employment) are percentages with scalar "units". Multiplying every `VALUE` by 1000 would break the rates.

## 8. Statistics vs Data type

In [11]:
pd.crosstab(df["Statistics"], df["Data type"])

Data type,Seasonally adjusted,Trend-cycle,Unadjusted
Statistics,,,
Estimate,1095008,6688,1457376
Standard error of estimate,969152,0,0
Standard error of month-to-month change,969152,0,0
Standard error of year-over-year change,969152,0,0


- Standard errors only exist for seasonally adjusted data.
- Trend-cycle is a single small slice: total employment, both genders, 15+, per province.

## 9. Missing values and STATUS

In [12]:
df["STATUS"].value_counts(dropna=False)

STATUS
NaN    4017756
..     1445758
x         3014
Name: count, dtype: int64

In [13]:
print("rows with null VALUE:", df["VALUE"].isna().sum())
print("null VALUE without STATUS:", (df["VALUE"].isna() & df["STATUS"].isna()).sum())
print("VALUE present with STATUS:", (df["VALUE"].notna() & df["STATUS"].notna()).sum())

rows with null VALUE: 1448772
null VALUE without STATUS: 0
VALUE present with STATUS: 0


In [14]:
null_share = df["VALUE"].isna().groupby(df["Statistics"], observed=True).mean()
null_share.round(3)

Statistics
Estimate                                   0.001
Standard error of estimate                 0.493
Standard error of month-to-month change    0.495
Standard error of year-over-year change    0.503
Name: VALUE, dtype: float64

In [15]:
first_valid = df[df["VALUE"].notna()].groupby("Statistics", observed=True)["REF_DATE"].min()
first_valid

Statistics
Estimate                                   1976-01
Standard error of estimate                 2001-01
Standard error of month-to-month change    2001-02
Standard error of year-over-year change    2001-07
Name: REF_DATE, dtype: str

In [16]:
estimates = df[df["Statistics"] == "Estimate"]
suppressed = estimates[estimates["STATUS"] == "x"]
suppressed.groupby(["GEO", "Age group"], observed=True).size().sort_values(ascending=False).head(10)

GEO                        Age group        
Newfoundland and Labrador  55 years and over    677
Alberta                    55 years and over    610
Prince Edward Island       55 years and over    494
Saskatchewan               55 years and over    324
New Brunswick              55 years and over    210
Nova Scotia                55 years and over    152
Manitoba                   55 years and over    110
Newfoundland and Labrador  15 to 19 years       110
Prince Edward Island       20 to 24 years       100
                           15 to 19 years        91
dtype: int64

- `STATUS` is only filled when `VALUE` is null, so it explains why the value is missing.
- `..` = not available for that period. Almost all of it is standard errors, which only start in 2001.
- `x` = suppressed for confidentiality. Happens in estimates for small groups (55+ in small provinces).
- Estimates themselves are ~99.9% complete.

## 10. Metadata file

Not a regular CSV: several tables stacked with different headers. Reading as text.

In [17]:
with zipfile.ZipFile(ZIP_PATH) as z:
    meta_text = z.read(META_FILE).decode("utf-8-sig")

start = meta_text.index("Symbol Legend")
end = meta_text.index("Survey Code")
print(meta_text[start:end])

Symbol Legend
Description,Symbol
"not available for a specific reference period","..",
"less than the limit of detection","<LOD",
"value rounded to 0 (zero) where there is a meaningful distinction between true zero and the value that was rounded","0s",
"data quality: excellent","A",
"data quality: very good","B",
"data quality: good","C",
"data quality: acceptable","D",
"use with caution","E",
"too unreliable to be published","F",
"not applicable","...",
"preliminary","p",
"revised","r",
"suppressed to meet the confidentiality requirements of the Statistics Act","x",
"terminated","t",

"


## 11. Constant and redundant columns

In [18]:
constant = [col for col in df.columns if df[col].nunique(dropna=False) == 1]
print("constant columns:", constant)

pairs = [("GEO", "DGUID"), ("UOM", "UOM_ID"), ("SCALAR_FACTOR", "SCALAR_ID"), ("VECTOR", "COORDINATE")]
for a, b in pairs:
    one_to_one = df.groupby(a, observed=True)[b].nunique().max() == 1
    print(f"{a} -> {b} is 1:1: {one_to_one}")

constant columns: ['SYMBOL', 'TERMINATED', 'DECIMALS']
GEO -> DGUID is 1:1: True


UOM -> UOM_ID is 1:1: True
SCALAR_FACTOR -> SCALAR_ID is 1:1: True


VECTOR -> COORDINATE is 1:1: True


## 12. Conclusions (spec for Bronze and Silver)

**Size and shape**
- ~5.5M rows, 19 columns, 8,991 series x 608 months (1976-01 to latest).
- Full history is republished every month; the file always contains every month again.

**Grain / key**
- One row = one series (`VECTOR`) in one month (`REF_DATE`). `VECTOR + REF_DATE` is unique.

**Bronze**
- Load the CSV as is, all columns as text. No filtering, no casting.
- **Load strategy: full refresh.** Every run replaces the whole table with the latest file.
  - StatCan republishes the full history every month, so the latest file is always a complete snapshot.
  - Past months get revised: seasonal factors are recalculated every year and earlier estimates can change. An incremental load (only appending new months) would keep the old, outdated values.
  - At ~5.5M rows, reloading everything is cheap. No need for incremental logic or merge keys.

**Silver: columns to keep**
- `REF_DATE` (cast to date, first day of month), `GEO`, `Labour force characteristics`, `Gender`, `Age group`, `Statistics`, `Data type`, `UOM`, `VALUE`, `STATUS`, `VECTOR`
- Rename to snake_case (e.g. `labour_force_characteristic`, `age_group`).

**Silver: columns to drop**
- Always empty: `SYMBOL`, `TERMINATED`
- Constant: `DECIMALS` (always 1)
- Redundant IDs: `DGUID`, `UOM_ID`, `SCALAR_ID`, `COORDINATE`
- `SCALAR_FACTOR` after applying the scale.

**Silver: scaling rule**
- `SCALAR_FACTOR = 'thousands'` -> `VALUE * 1000` (person counts)
- `SCALAR_FACTOR = 'units'` -> keep `VALUE` (percentages)

**Missing values**
- `VALUE` null always comes with a `STATUS` (`..` not available, `x` suppressed). Keep the nulls, keep `STATUS` so it's clear why.

**Silver keeps every row**
- Silver only cleans: renames, casts, applies the scale, drops useless columns. No business filters.
- Filters live in Gold, so each Gold model picks what it needs and nothing is lost upstream.

**Gold filters (decided)**
- `Statistics = 'Estimate'` only. Standard errors are ~50% null and only start in 2001.
- `Data type = 'Seasonally adjusted'` as the default, the standard for month-to-month comparisons.
- `Unadjusted` only if a model needs year-over-year comparisons (same month across years).
- `Trend-cycle` dropped: tiny slice, total employment only.